In [3]:
import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_squared_error, make_scorer

from catboost import CatBoostRegressor


# =========================================================
# CONFIG
# =========================================================
DATA_PATH = "data_rdkit.csv"
MODEL_BUNDLE_PATH = "catboost_pipeline_bundle.pkl"

SMILES_COL = "Smiles"
TARGET_COL = "pIC50"

MODEL_PARAMS = dict(
    iterations=1000,
    learning_rate=0.1,
    depth=6,
    eval_metric="R2",
    verbose=100,
    random_seed=42
)


# =========================================================
# UTILS
# =========================================================
class ColumnSelector(BaseEstimator, TransformerMixin):
    """
    Выбирает только нужные колонки и сохраняет порядок признаков.
    Никакой логики модели не меняет.
    """
    def __init__(self, columns):
        self.columns = columns

    def fit(self, X, y=None):
        missing = [c for c in self.columns if c not in X.columns]
        if missing:
            raise ValueError(f"Missing columns in input DataFrame: {missing}")
        return self

    def transform(self, X):
        return X.loc[:, self.columns]


def rmse_func(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))


# =========================================================
# LOAD DATA
# =========================================================
df = pd.read_csv(DATA_PATH)
df = df.dropna().copy()

feature_columns = [c for c in df.columns if c not in [SMILES_COL, TARGET_COL]]

X = df[feature_columns].copy()
y = df[TARGET_COL].values

print(f"Rows: {len(df)}")
print(f"Features: {len(feature_columns)}")
print("Feature columns:")
print(feature_columns)


#========================================================
from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold


def get_scaffold(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return MurckoScaffold.MurckoScaffoldSmiles(mol=mol)


df["scaffold"] = df[SMILES_COL].apply(get_scaffold)

print("Unique scaffolds:", df["scaffold"].nunique())
from sklearn.model_selection import GroupKFold


# =========================================================
# TRAIN / TEST SPLIT
# =========================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)


# =========================================================
# PIPELINE
# =========================================================
pipeline = Pipeline([
    ("select_columns", ColumnSelector(feature_columns)),
    ("model", CatBoostRegressor(**MODEL_PARAMS))
])


# =========================================================
# CROSS-VALIDATION
# =========================================================
print("=" * 50)
print("RANDOM CV")
print("=" * 50)

cv_r2_random = cross_val_score(
    pipeline,
    X_train,
    y_train,
    cv=5,
    scoring="r2"
)

print(f"Random CV R2 mean: {cv_r2_random.mean():.4f}")
print(f"Random CV R2 std:  {cv_r2_random.std():.4f}")


print("\n" + "=" * 50)
print("SCAFFOLD CV")
print("=" * 50)

gkf = GroupKFold(n_splits=5)

cv_r2_scaffold = cross_val_score(
    pipeline,
    X,
    y,
    groups=df["scaffold"],
    cv=gkf,
    scoring="r2"
)

print(f"Scaffold CV R2 mean: {cv_r2_scaffold.mean():.4f}")
print(f"Scaffold CV R2 std:  {cv_r2_scaffold.std():.4f}")

# =========================================================
# FIT
# =========================================================
pipeline.fit(X_train, y_train)


# =========================================================
# EVALUATION
# =========================================================
y_pred = pipeline.predict(X_test)

r2 = r2_score(y_test, y_pred)
rmse = rmse_func(y_test, y_pred)

print("\n" + "=" * 50)
print("TEST METRICS")
print("=" * 50)
print(f"R2 Score: {r2:.4f}")
print(f"RMSE:     {rmse:.4f}")


# =========================================================
# FEATURE IMPORTANCE
# =========================================================
model = pipeline.named_steps["model"]
importances = model.get_feature_importance()

importance_df = pd.DataFrame({
    "feature": feature_columns,
    "importance": importances
}).sort_values("importance", ascending=False)

print("\nFeature importance:")
print(importance_df)


# =========================================================
# SAVE BUNDLE
# =========================================================
bundle = {
    "pipeline": pipeline,
    "feature_columns": feature_columns,
    "smiles_col": SMILES_COL,
    "target_col": TARGET_COL,
    "model_params": MODEL_PARAMS,
    "logic_note": "Features are all columns except Smiles and pIC50"
}

with open(MODEL_BUNDLE_PATH, "wb") as f:
    pickle.dump(bundle, f)

print(f"\nSaved bundle to: {MODEL_BUNDLE_PATH}")

Rows: 12358
Features: 164
Feature columns:
['MaxAbsEStateIndex', 'MinAbsEStateIndex', 'MinEStateIndex', 'qed', 'SPS', 'MolWt', 'MaxPartialCharge', 'MinPartialCharge', 'FpDensityMorgan1', 'BCUT2D_MWHI', 'BCUT2D_MWLOW', 'BCUT2D_CHGHI', 'BCUT2D_CHGLO', 'BCUT2D_LOGPHI', 'BCUT2D_LOGPLOW', 'BCUT2D_MRHI', 'BCUT2D_MRLOW', 'AvgIpc', 'BalabanJ', 'BertzCT', 'HallKierAlpha', 'PEOE_VSA1', 'PEOE_VSA10', 'PEOE_VSA11', 'PEOE_VSA12', 'PEOE_VSA13', 'PEOE_VSA14', 'PEOE_VSA2', 'PEOE_VSA3', 'PEOE_VSA4', 'PEOE_VSA5', 'PEOE_VSA6', 'PEOE_VSA7', 'PEOE_VSA8', 'PEOE_VSA9', 'SMR_VSA1', 'SMR_VSA10', 'SMR_VSA2', 'SMR_VSA3', 'SMR_VSA4', 'SMR_VSA5', 'SMR_VSA6', 'SMR_VSA7', 'SMR_VSA9', 'SlogP_VSA1', 'SlogP_VSA10', 'SlogP_VSA11', 'SlogP_VSA12', 'SlogP_VSA2', 'SlogP_VSA3', 'SlogP_VSA4', 'SlogP_VSA5', 'SlogP_VSA7', 'SlogP_VSA8', 'TPSA', 'EState_VSA1', 'EState_VSA10', 'EState_VSA11', 'EState_VSA2', 'EState_VSA3', 'EState_VSA4', 'EState_VSA5', 'EState_VSA6', 'EState_VSA7', 'EState_VSA8', 'EState_VSA9', 'VSA_EState1', 'VSA_

In [4]:
print(cv_r2_scaffold)

[0.66285353 0.66363134 0.62833016 0.65556855 0.48580822]


In [8]:
import numpy as np
import pandas as pd

from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem
from rdkit.Chem.Scaffolds import MurckoScaffold

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import VarianceThreshold
from sklearn.model_selection import cross_val_score, GroupKFold, KFold
from sklearn.metrics import make_scorer, mean_squared_error
from catboost import CatBoostRegressor


# =========================================================
# CONFIG
# =========================================================
DATA_PATH = "data_rdkit.csv"

SMILES_COL = "Smiles"
TARGET_COL = "pIC50"

FP_RADIUS = 3
FP_NBITS = 1024
FP_PREFIX = "fp_"

MODEL_PARAMS = dict(
    iterations=1000,
    learning_rate=0.1,
    depth=6,
    eval_metric="R2",
    verbose=0,
    random_seed=42
)


# =========================================================
# HELPERS
# =========================================================
def rmse_func(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))


def get_scaffold(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return "INVALID"
    return MurckoScaffold.MurckoScaffoldSmiles(mol=mol)


def morgan_fp_bits(smiles, radius=2, n_bits=2048):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return [0] * n_bits
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
    arr = np.zeros((n_bits,), dtype=np.int8)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr.tolist()


def build_pipeline(desc_cols, fp_cols, model_params):
    transformers = []

    if len(desc_cols) > 0:
        desc_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="median"))
        ])
        transformers.append(("desc", desc_pipe, desc_cols))

    if len(fp_cols) > 0:
        fp_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ("variance", VarianceThreshold(threshold=0.0))
        ])
        transformers.append(("fp", fp_pipe, fp_cols))

    preprocessor = ColumnTransformer(transformers, remainder="drop")

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", CatBoostRegressor(**model_params))
    ])
    return pipeline


def evaluate_mode(X, y, groups, desc_cols, fp_cols, mode_name):
    pipeline = build_pipeline(desc_cols, fp_cols, MODEL_PARAMS)

    random_cv = KFold(n_splits=5, shuffle=True, random_state=42)
    scaffold_cv = GroupKFold(n_splits=5)

    random_r2 = cross_val_score(
        pipeline,
        X,
        y,
        cv=random_cv,
        scoring="r2",
        n_jobs=-1
    )

    scaffold_r2 = cross_val_score(
        pipeline,
        X,
        y,
        groups=groups,
        cv=scaffold_cv,
        scoring="r2",
        n_jobs=-1
    )

    rmse_scorer = make_scorer(
        lambda yt, yp: np.sqrt(mean_squared_error(yt, yp)),
        greater_is_better=False
    )

    random_rmse = -cross_val_score(
        pipeline,
        X,
        y,
        cv=random_cv,
        scoring=rmse_scorer,
        n_jobs=-1
    )

    scaffold_rmse = -cross_val_score(
        pipeline,
        X,
        y,
        groups=groups,
        cv=scaffold_cv,
        scoring=rmse_scorer,
        n_jobs=-1
    )

    return {
        "mode": mode_name,
        "n_desc": len(desc_cols),
        "n_fp": len(fp_cols),
        "n_total": len(desc_cols) + len(fp_cols),
        "random_r2_mean": random_r2.mean(),
        "random_r2_std": random_r2.std(),
        "scaffold_r2_mean": scaffold_r2.mean(),
        "scaffold_r2_std": scaffold_r2.std(),
        "random_rmse_mean": random_rmse.mean(),
        "random_rmse_std": random_rmse.std(),
        "scaffold_rmse_mean": scaffold_rmse.mean(),
        "scaffold_rmse_std": scaffold_rmse.std(),
        "random_r2_folds": random_r2,
        "scaffold_r2_folds": scaffold_r2
    }


# =========================================================
# LOAD DATA
# =========================================================
df = pd.read_csv(DATA_PATH).dropna().copy()

print(f"Initial rows: {len(df)}")

# scaffold groups
df["scaffold"] = df[SMILES_COL].apply(get_scaffold)

# descriptors already present in CSV
descriptor_columns = [
    c for c in df.columns
    if c not in [SMILES_COL, TARGET_COL, "scaffold"]
]

print(f"Descriptor features found in CSV: {len(descriptor_columns)}")

# =========================================================
# CALCULATE FINGERPRINTS
# =========================================================
print("Calculating Morgan fingerprints...")

fp_matrix = df[SMILES_COL].apply(
    lambda smi: morgan_fp_bits(smi, radius=FP_RADIUS, n_bits=FP_NBITS)
)

fp_df = pd.DataFrame(
    fp_matrix.tolist(),
    columns=[f"{FP_PREFIX}{i}" for i in range(FP_NBITS)],
    index=df.index
)

df = pd.concat([df, fp_df], axis=1)

fingerprint_columns = list(fp_df.columns)

print(f"Fingerprint features added: {len(fingerprint_columns)}")
print(f"Unique scaffolds: {df['scaffold'].nunique()}")
print(df["scaffold"].value_counts().describe())


# =========================================================
# PREPARE MATRICES
# =========================================================
all_feature_columns = descriptor_columns + fingerprint_columns

X = df[all_feature_columns].copy()
y = df[TARGET_COL].values
groups = df["scaffold"].values


# =========================================================
# RUN 3 EXPERIMENTS
# =========================================================
results = []

experiments = [
    ("descriptors_only", descriptor_columns, []),
    ("fingerprints_only", [], fingerprint_columns),
    ("descriptors_plus_fingerprints", descriptor_columns, fingerprint_columns),
]

for mode_name, desc_cols, fp_cols in experiments:
    print("\n" + "=" * 70)
    print(f"Running: {mode_name}")
    print("=" * 70)

    res = evaluate_mode(
        X=X,
        y=y,
        groups=groups,
        desc_cols=desc_cols,
        fp_cols=fp_cols,
        mode_name=mode_name
    )
    results.append(res)

    print(f"Random CV R2:    {res['random_r2_mean']:.4f} ± {res['random_r2_std']:.4f}")
    print(f"Scaffold CV R2:  {res['scaffold_r2_mean']:.4f} ± {res['scaffold_r2_std']:.4f}")
    print(f"Random CV RMSE:  {res['random_rmse_mean']:.4f} ± {res['random_rmse_std']:.4f}")
    print(f"Scaffold CV RMSE:{res['scaffold_rmse_mean']:.4f} ± {res['scaffold_rmse_std']:.4f}")
    print(f"Random R2 folds:   {np.round(res['random_r2_folds'], 4)}")
    print(f"Scaffold R2 folds: {np.round(res['scaffold_r2_folds'], 4)}")


# =========================================================
# SUMMARY TABLE
# =========================================================
results_df = pd.DataFrame(results)[[
    "mode",
    "n_desc",
    "n_fp",
    "n_total",
    "random_r2_mean",
    "random_r2_std",
    "scaffold_r2_mean",
    "scaffold_r2_std",
    "random_rmse_mean",
    "random_rmse_std",
    "scaffold_rmse_mean",
    "scaffold_rmse_std"
]].sort_values("scaffold_r2_mean", ascending=False)

print("\n" + "=" * 70)
print("FINAL COMPARISON")
print("=" * 70)
print(results_df.to_string(index=False))

Initial rows: 12358
Descriptor features found in CSV: 164
Calculating Morgan fingerprints...


[04:01:33] DEPRECATION WARNING: please use MorganGenerator
[04:01:33] DEPRECATION WARNING: please use MorganGenerator
[04:01:33] DEPRECATION WARNING: please use MorganGenerator
[04:01:33] DEPRECATION WARNING: please use MorganGenerator
[04:01:33] DEPRECATION WARNING: please use MorganGenerator
[04:01:33] DEPRECATION WARNING: please use MorganGenerator
[04:01:33] DEPRECATION WARNING: please use MorganGenerator
[04:01:33] DEPRECATION WARNING: please use MorganGenerator
[04:01:33] DEPRECATION WARNING: please use MorganGenerator
[04:01:33] DEPRECATION WARNING: please use MorganGenerator
[04:01:33] DEPRECATION WARNING: please use MorganGenerator
[04:01:33] DEPRECATION WARNING: please use MorganGenerator
[04:01:33] DEPRECATION WARNING: please use MorganGenerator
[04:01:33] DEPRECATION WARNING: please use MorganGenerator
[04:01:33] DEPRECATION WARNING: please use MorganGenerator
[04:01:33] DEPRECATION WARNING: please use MorganGenerator
[04:01:33] DEPRECATION WARNING: please use MorganGenerat

Fingerprint features added: 1024
Unique scaffolds: 4735
count    4735.000000
mean        2.609926
std         7.393270
min         1.000000
25%         1.000000
50%         1.000000
75%         2.000000
max       272.000000
Name: count, dtype: float64

Running: descriptors_only
Random CV R2:    0.7600 ± 0.0134
Scaffold CV R2:  0.6192 ± 0.0679
Random CV RMSE:  0.7141 ± 0.0206
Scaffold CV RMSE:0.8967 ± 0.1218
Random R2 folds:   [0.7649 0.7762 0.7474 0.7412 0.7702]
Scaffold R2 folds: [0.6629 0.6636 0.6283 0.6556 0.4858]

Running: fingerprints_only
Random CV R2:    0.7688 ± 0.0056
Scaffold CV R2:  0.6429 ± 0.0602
Random CV RMSE:  0.7011 ± 0.0094
Scaffold CV RMSE:0.8677 ± 0.1081
Random R2 folds:   [0.767  0.7758 0.7662 0.7607 0.7741]
Scaffold R2 folds: [0.6631 0.6689 0.6517 0.7031 0.5275]

Running: descriptors_plus_fingerprints
Random CV R2:    0.7812 ± 0.0125
Scaffold CV R2:  0.6639 ± 0.0615
Random CV RMSE:  0.6818 ± 0.0192
Scaffold CV RMSE:0.8421 ± 0.1145
Random R2 folds:   [0.7832 0.7962

In [ ]:
import numpy as np
import pandas as pd
import optuna

from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem
from rdkit.Chem.Scaffolds import MurckoScaffold

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import VarianceThreshold
from sklearn.model_selection import cross_validate, GroupKFold, KFold
from sklearn.metrics import make_scorer, mean_squared_error
from catboost import CatBoostRegressor


# =========================================================
# CONFIG
# =========================================================
DATA_PATH = "data_rdkit.csv"

SMILES_COL = "Smiles"
TARGET_COL = "pIC50"

FP_RADIUS = 3
FP_NBITS = 1024
FP_PREFIX = "fp_"

RANDOM_STATE = 42

MODEL_PARAMS = dict(
    iterations=1000,
    learning_rate=0.1,
    depth=6,
    eval_metric="R2",
    verbose=0,
    random_seed=RANDOM_STATE
)

# --- Optuna config ---
USE_OPTUNA = True
OPTUNA_TRIALS = 15
OPTUNA_CV_SPLITS = 3
OPTUNA_TARGET_MODE = "descriptors_plus_fingerprints"  
# варианты:
# "descriptors_only"
# "fingerprints_only"
# "descriptors_plus_fingerprints"


# =========================================================
# HELPERS
# =========================================================
def rmse_func(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))


def get_scaffold(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return "INVALID"
    return MurckoScaffold.MurckoScaffoldSmiles(mol=mol)


def morgan_fp_bits(smiles, radius=2, n_bits=2048):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return [0] * n_bits
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
    arr = np.zeros((n_bits,), dtype=np.int8)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr.tolist()


def build_pipeline(desc_cols, fp_cols, model_params):
    transformers = []

    if len(desc_cols) > 0:
        desc_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="median"))
        ])
        transformers.append(("desc", desc_pipe, desc_cols))

    if len(fp_cols) > 0:
        fp_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ("variance", VarianceThreshold(threshold=0.0))
        ])
        transformers.append(("fp", fp_pipe, fp_cols))

    preprocessor = ColumnTransformer(transformers, remainder="drop")

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", CatBoostRegressor(**model_params))
    ])
    return pipeline


def get_cv_scores(pipeline, X, y, groups=None, n_splits=5, mode="random"):
    rmse_scorer = make_scorer(rmse_func, greater_is_better=False)

    if mode == "random":
        cv = KFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
        cv_result = cross_validate(
            pipeline,
            X,
            y,
            cv=cv,
            scoring={"r2": "r2", "rmse": rmse_scorer},
            n_jobs=-1,
            return_train_score=False
        )
    elif mode == "scaffold":
        cv = GroupKFold(n_splits=n_splits)
        cv_result = cross_validate(
            pipeline,
            X,
            y,
            groups=groups,
            cv=cv,
            scoring={"r2": "r2", "rmse": rmse_scorer},
            n_jobs=-1,
            return_train_score=False
        )
    else:
        raise ValueError(f"Unknown mode: {mode}")

    r2_scores = cv_result["test_r2"]
    rmse_scores = -cv_result["test_rmse"]

    return r2_scores, rmse_scores


def evaluate_mode(X, y, groups, desc_cols, fp_cols, mode_name, model_params):
    pipeline = build_pipeline(desc_cols, fp_cols, model_params)

    random_r2, random_rmse = get_cv_scores(
        pipeline, X, y, groups=None, n_splits=5, mode="random"
    )

    scaffold_r2, scaffold_rmse = get_cv_scores(
        pipeline, X, y, groups=groups, n_splits=5, mode="scaffold"
    )

    return {
        "mode": mode_name,
        "n_desc": len(desc_cols),
        "n_fp": len(fp_cols),
        "n_total": len(desc_cols) + len(fp_cols),
        "random_r2_mean": random_r2.mean(),
        "random_r2_std": random_r2.std(),
        "scaffold_r2_mean": scaffold_r2.mean(),
        "scaffold_r2_std": scaffold_r2.std(),
        "random_rmse_mean": random_rmse.mean(),
        "random_rmse_std": random_rmse.std(),
        "scaffold_rmse_mean": scaffold_rmse.mean(),
        "scaffold_rmse_std": scaffold_rmse.std(),
        "random_r2_folds": random_r2,
        "scaffold_r2_folds": scaffold_r2
    }


def optimize_catboost_optuna(X, y, groups, desc_cols, fp_cols, n_trials=15, cv_splits=3):
    def objective(trial):
        params = dict(
            iterations=trial.suggest_int("iterations", 300, 1000, step=100),
            learning_rate=trial.suggest_float("learning_rate", 0.02, 0.2, log=True),
            depth=trial.suggest_int("depth", 4, 8),
            l2_leaf_reg=trial.suggest_float("l2_leaf_reg", 1.0, 10.0, log=True),
            random_strength=trial.suggest_float("random_strength", 1e-3, 5.0, log=True),
            bagging_temperature=trial.suggest_float("bagging_temperature", 0.0, 2.0),
            eval_metric="R2",
            verbose=0,
            random_seed=RANDOM_STATE
        )

        pipeline = build_pipeline(desc_cols, fp_cols, params)

        cv = GroupKFold(n_splits=cv_splits)

        scores = cross_validate(
            pipeline,
            X,
            y,
            groups=groups,
            cv=cv,
            scoring={"r2": "r2"},
            n_jobs=-1,
            return_train_score=False
        )

        return float(scores["test_r2"].mean())

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)

    best_params = study.best_params.copy()
    best_params.update({
        "eval_metric": "R2",
        "verbose": 0,
        "random_seed": RANDOM_STATE
    })

    return study, best_params


# =========================================================
# LOAD DATA
# =========================================================
df = pd.read_csv(DATA_PATH).dropna().copy()

print(f"Initial rows: {len(df)}")

# scaffold groups
df["scaffold"] = df[SMILES_COL].apply(get_scaffold)

# descriptors already present in CSV
descriptor_columns = [
    c for c in df.columns
    if c not in [SMILES_COL, TARGET_COL, "scaffold"]
]

print(f"Descriptor features found in CSV: {len(descriptor_columns)}")

# =========================================================
# CALCULATE FINGERPRINTS
# =========================================================
print("Calculating Morgan fingerprints...")

fp_matrix = df[SMILES_COL].apply(
    lambda smi: morgan_fp_bits(smi, radius=FP_RADIUS, n_bits=FP_NBITS)
)

fp_df = pd.DataFrame(
    fp_matrix.tolist(),
    columns=[f"{FP_PREFIX}{i}" for i in range(FP_NBITS)],
    index=df.index
)

df = pd.concat([df, fp_df], axis=1)

fingerprint_columns = list(fp_df.columns)

print(f"Fingerprint features added: {len(fingerprint_columns)}")
print(f"Unique scaffolds: {df['scaffold'].nunique()}")
print(df["scaffold"].value_counts().describe())


# =========================================================
# PREPARE MATRICES
# =========================================================
all_feature_columns = descriptor_columns + fingerprint_columns

X = df[all_feature_columns].copy()
y = df[TARGET_COL].values
groups = df["scaffold"].values


# =========================================================
# RUN 3 EXPERIMENTS WITH BASE PARAMS
# =========================================================
results = []

experiments = [
    ("descriptors_only", descriptor_columns, []),
    ("fingerprints_only", [], fingerprint_columns),
    ("descriptors_plus_fingerprints", descriptor_columns, fingerprint_columns),
]

for mode_name, desc_cols, fp_cols in experiments:
    print("\n" + "=" * 70)
    print(f"Running: {mode_name}")
    print("=" * 70)

    res = evaluate_mode(
        X=X,
        y=y,
        groups=groups,
        desc_cols=desc_cols,
        fp_cols=fp_cols,
        mode_name=mode_name,
        model_params=MODEL_PARAMS
    )
    results.append(res)

    print(f"Random CV R2:     {res['random_r2_mean']:.4f} ± {res['random_r2_std']:.4f}")
    print(f"Scaffold CV R2:   {res['scaffold_r2_mean']:.4f} ± {res['scaffold_r2_std']:.4f}")
    print(f"Random CV RMSE:   {res['random_rmse_mean']:.4f} ± {res['random_rmse_std']:.4f}")
    print(f"Scaffold CV RMSE: {res['scaffold_rmse_mean']:.4f} ± {res['scaffold_rmse_std']:.4f}")
    print(f"Random R2 folds:   {np.round(res['random_r2_folds'], 4)}")
    print(f"Scaffold R2 folds: {np.round(res['scaffold_r2_folds'], 4)}")


# =========================================================
# SUMMARY TABLE
# =========================================================
results_df = pd.DataFrame(results)[[
    "mode",
    "n_desc",
    "n_fp",
    "n_total",
    "random_r2_mean",
    "random_r2_std",
    "scaffold_r2_mean",
    "scaffold_r2_std",
    "random_rmse_mean",
    "random_rmse_std",
    "scaffold_rmse_mean",
    "scaffold_rmse_std"
]].sort_values("scaffold_r2_mean", ascending=False)

print("\n" + "=" * 70)
print("FINAL COMPARISON (BASE PARAMS)")
print("=" * 70)
print(results_df.to_string(index=False))


# =========================================================
# OPTUNA
# =========================================================
if USE_OPTUNA:
    experiment_dict = {
        "descriptors_only": (descriptor_columns, []),
        "fingerprints_only": ([], fingerprint_columns),
        "descriptors_plus_fingerprints": (descriptor_columns, fingerprint_columns),
    }

    desc_cols_opt, fp_cols_opt = experiment_dict[OPTUNA_TARGET_MODE]

    print("\n" + "=" * 70)
    print(f"OPTUNA FOR: {OPTUNA_TARGET_MODE}")
    print("=" * 70)

    study, best_params = optimize_catboost_optuna(
        X=X,
        y=y,
        groups=groups,
        desc_cols=desc_cols_opt,
        fp_cols=fp_cols_opt,
        n_trials=OPTUNA_TRIALS,
        cv_splits=OPTUNA_CV_SPLITS
    )

    print(f"Best Optuna scaffold CV R2: {study.best_value:.4f}")
    print("Best params:")
    for k, v in best_params.items():
        print(f"  {k}: {v}")

    print("\n" + "=" * 70)
    print(f"RE-EVALUATION WITH OPTUNA PARAMS: {OPTUNA_TARGET_MODE}")
    print("=" * 70)

    tuned_res = evaluate_mode(
        X=X,
        y=y,
        groups=groups,
        desc_cols=desc_cols_opt,
        fp_cols=fp_cols_opt,
        mode_name=f"{OPTUNA_TARGET_MODE}_optuna",
        model_params=best_params
    )

    print(f"Random CV R2:     {tuned_res['random_r2_mean']:.4f} ± {tuned_res['random_r2_std']:.4f}")
    print(f"Scaffold CV R2:   {tuned_res['scaffold_r2_mean']:.4f} ± {tuned_res['scaffold_r2_std']:.4f}")
    print(f"Random CV RMSE:   {tuned_res['random_rmse_mean']:.4f} ± {tuned_res['random_rmse_std']:.4f}")
    print(f"Scaffold CV RMSE: {tuned_res['scaffold_rmse_mean']:.4f} ± {tuned_res['scaffold_rmse_std']:.4f}")
    print(f"Random R2 folds:   {np.round(tuned_res['random_r2_folds'], 4)}")
    print(f"Scaffold R2 folds: {np.round(tuned_res['scaffold_r2_folds'], 4)}")

Initial rows: 12358
Descriptor features found in CSV: 164
Calculating Morgan fingerprints...


[08:24:03] DEPRECATION WARNING: please use MorganGenerator
[08:24:03] DEPRECATION WARNING: please use MorganGenerator
[08:24:03] DEPRECATION WARNING: please use MorganGenerator
[08:24:03] DEPRECATION WARNING: please use MorganGenerator
[08:24:03] DEPRECATION WARNING: please use MorganGenerator
[08:24:03] DEPRECATION WARNING: please use MorganGenerator
[08:24:03] DEPRECATION WARNING: please use MorganGenerator
[08:24:03] DEPRECATION WARNING: please use MorganGenerator
[08:24:03] DEPRECATION WARNING: please use MorganGenerator
[08:24:03] DEPRECATION WARNING: please use MorganGenerator
[08:24:03] DEPRECATION WARNING: please use MorganGenerator
[08:24:03] DEPRECATION WARNING: please use MorganGenerator
[08:24:03] DEPRECATION WARNING: please use MorganGenerator
[08:24:03] DEPRECATION WARNING: please use MorganGenerator
[08:24:03] DEPRECATION WARNING: please use MorganGenerator
[08:24:03] DEPRECATION WARNING: please use MorganGenerator
[08:24:03] DEPRECATION WARNING: please use MorganGenerat

Fingerprint features added: 1024
Unique scaffolds: 4735
count    4735.000000
mean        2.609926
std         7.393270
min         1.000000
25%         1.000000
50%         1.000000
75%         2.000000
max       272.000000
Name: count, dtype: float64

Running: descriptors_only
Random CV R2:     0.7600 ± 0.0134
Scaffold CV R2:   0.6192 ± 0.0679
Random CV RMSE:   0.7141 ± 0.0206
Scaffold CV RMSE: 0.8967 ± 0.1218
Random R2 folds:   [0.7649 0.7762 0.7474 0.7412 0.7702]
Scaffold R2 folds: [0.6629 0.6636 0.6283 0.6556 0.4858]

Running: fingerprints_only
Random CV R2:     0.7688 ± 0.0056
Scaffold CV R2:   0.6429 ± 0.0602
Random CV RMSE:   0.7011 ± 0.0094
Scaffold CV RMSE: 0.8677 ± 0.1081
Random R2 folds:   [0.767  0.7758 0.7662 0.7607 0.7741]
Scaffold R2 folds: [0.6631 0.6689 0.6517 0.7031 0.5275]

Running: descriptors_plus_fingerprints


[I 2026-03-22 08:30:43,339] A new study created in memory with name: no-name-f682fb24-429e-4bbb-a454-b13bbbb073e1


Random CV R2:     0.7812 ± 0.0125
Scaffold CV R2:   0.6639 ± 0.0615
Random CV RMSE:   0.6818 ± 0.0192
Scaffold CV RMSE: 0.8421 ± 0.1145
Random R2 folds:   [0.7832 0.7962 0.7679 0.7658 0.7928]
Scaffold R2 folds: [0.7071 0.7039 0.6628 0.7007 0.5451]

FINAL COMPARISON (BASE PARAMS)
                         mode  n_desc  n_fp  n_total  random_r2_mean  random_r2_std  scaffold_r2_mean  scaffold_r2_std  random_rmse_mean  random_rmse_std  scaffold_rmse_mean  scaffold_rmse_std
descriptors_plus_fingerprints     164  1024     1188        0.781163       0.012466          0.663903         0.061529          0.681828         0.019214            0.842146           0.114535
            fingerprints_only       0  1024     1024        0.768753       0.005557          0.642865         0.060169          0.701146         0.009361            0.867732           0.108116
             descriptors_only     164     0      164        0.759988       0.013431          0.619238         0.067940          0.714091     

[I 2026-03-22 08:30:52,007] Trial 0 finished with value: 0.5115575831665722 and parameters: {'iterations': 300, 'learning_rate': 0.03583222255620342, 'depth': 4, 'l2_leaf_reg': 1.7865694396505123, 'random_strength': 0.009812269916891051, 'bagging_temperature': 0.08248006349859338}. Best is trial 0 with value: 0.5115575831665722.
[I 2026-03-22 08:31:54,548] Trial 1 finished with value: 0.6371959058107529 and parameters: {'iterations': 600, 'learning_rate': 0.10124416246040052, 'depth': 7, 'l2_leaf_reg': 6.538527135043493, 'random_strength': 0.013661142141169983, 'bagging_temperature': 1.4923019110613316}. Best is trial 1 with value: 0.6371959058107529.
[I 2026-03-22 08:32:29,150] Trial 2 finished with value: 0.6160858848636294 and parameters: {'iterations': 1000, 'learning_rate': 0.04033290649645562, 'depth': 5, 'l2_leaf_reg': 7.184741859859763, 'random_strength': 0.15880172887419894, 'bagging_temperature': 1.3215969261234886}. Best is trial 1 with value: 0.6371959058107529.
[I 2026-03-

Best Optuna scaffold CV R2: 0.6524
Best params:
  iterations: 900
  learning_rate: 0.05178968283138356
  depth: 8
  l2_leaf_reg: 3.6489631502568898
  random_strength: 3.8782490985210782
  bagging_temperature: 0.6980163454642099
  eval_metric: R2
  verbose: 0
  random_seed: 42

RE-EVALUATION WITH OPTUNA PARAMS: descriptors_plus_fingerprints


In [9]:
import pickle
import numpy as np
import pandas as pd

from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem
from rdkit.Chem.Scaffolds import MurckoScaffold

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import VarianceThreshold
from sklearn.model_selection import GroupKFold

from catboost import CatBoostRegressor


# =========================================================
# CONFIG
# =========================================================
DATA_PATH = "data_rdkit.csv"
ENSEMBLE_BUNDLE_PATH = "catboost_ensemble_bundle.pkl"

SMILES_COL = "Smiles"
TARGET_COL = "pIC50"

FP_RADIUS = 2
FP_NBITS = 2048
FP_PREFIX = "fp_"

N_ENSEMBLE_MODELS = 5

MODEL_PARAMS = dict(
    iterations=1000,
    learning_rate=0.1,
    depth=6,
    eval_metric="R2",
    verbose=0,
    random_seed=42
)


# =========================================================
# HELPERS
# =========================================================
def get_scaffold(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return "INVALID"
    return MurckoScaffold.MurckoScaffoldSmiles(mol=mol)


def morgan_fp_bits(smiles, radius=2, n_bits=2048):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return [0] * n_bits
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
    arr = np.zeros((n_bits,), dtype=np.int8)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr.tolist()


def morgan_fp_bitvect(smiles, radius=2, n_bits=2048):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)


def build_pipeline(desc_cols, fp_cols, model_params):
    transformers = []

    if len(desc_cols) > 0:
        desc_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="median"))
        ])
        transformers.append(("desc", desc_pipe, desc_cols))

    if len(fp_cols) > 0:
        fp_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ("variance", VarianceThreshold(threshold=0.0))
        ])
        transformers.append(("fp", fp_pipe, fp_cols))

    preprocessor = ColumnTransformer(transformers, remainder="drop")

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", CatBoostRegressor(**model_params))
    ])
    return pipeline

In [10]:
# =========================================================
# TRAIN ENSEMBLE
# =========================================================
base_pipeline = build_pipeline(
    descriptor_columns,
    fingerprint_columns,
    MODEL_PARAMS
)

gkf = GroupKFold(n_splits=N_ENSEMBLE_MODELS)

ensemble_models = []
fold_info = []

for fold_id, (train_idx, valid_idx) in enumerate(gkf.split(X, y, groups=groups), start=1):
    print(f"Training ensemble model {fold_id}/{N_ENSEMBLE_MODELS} ...")

    X_train_fold = X.iloc[train_idx]
    y_train_fold = y[train_idx]

    model = clone(base_pipeline)
    model.fit(X_train_fold, y_train_fold)

    ensemble_models.append(model)
    fold_info.append({
        "fold_id": fold_id,
        "train_size": len(train_idx),
        "valid_size": len(valid_idx)
    })

print("Ensemble training done.")

Training ensemble model 1/5 ...
Training ensemble model 2/5 ...
Training ensemble model 3/5 ...
Training ensemble model 4/5 ...
Training ensemble model 5/5 ...
Ensemble training done.


In [11]:
# =========================================================
# STORE TRAIN FINGERPRINTS FOR SIMILARITY-BASED UNCERTAINTY
# =========================================================
train_bitvects = [
    morgan_fp_bitvect(smi, radius=FP_RADIUS, n_bits=FP_NBITS)
    for smi in df[SMILES_COL]
]

bundle = {
    "ensemble_models": ensemble_models,
    "descriptor_columns": descriptor_columns,
    "fingerprint_columns": fingerprint_columns,
    "smiles_col": SMILES_COL,
    "target_col": TARGET_COL,
    "fp_radius": FP_RADIUS,
    "fp_nbits": FP_NBITS,
    "train_smiles": df[SMILES_COL].tolist(),
    "train_bitvects": train_bitvects,
    "fold_info": fold_info,
    "model_params": MODEL_PARAMS
}

with open(ENSEMBLE_BUNDLE_PATH, "wb") as f:
    pickle.dump(bundle, f)

print(f"Saved ensemble bundle to: {ENSEMBLE_BUNDLE_PATH}")

[04:14:50] DEPRECATION WARNING: please use MorganGenerator
[04:14:50] DEPRECATION WARNING: please use MorganGenerator
[04:14:50] DEPRECATION WARNING: please use MorganGenerator
[04:14:50] DEPRECATION WARNING: please use MorganGenerator
[04:14:50] DEPRECATION WARNING: please use MorganGenerator
[04:14:50] DEPRECATION WARNING: please use MorganGenerator
[04:14:50] DEPRECATION WARNING: please use MorganGenerator
[04:14:50] DEPRECATION WARNING: please use MorganGenerator
[04:14:50] DEPRECATION WARNING: please use MorganGenerator
[04:14:50] DEPRECATION WARNING: please use MorganGenerator
[04:14:50] DEPRECATION WARNING: please use MorganGenerator
[04:14:50] DEPRECATION WARNING: please use MorganGenerator
[04:14:50] DEPRECATION WARNING: please use MorganGenerator
[04:14:50] DEPRECATION WARNING: please use MorganGenerator
[04:14:50] DEPRECATION WARNING: please use MorganGenerator
[04:14:50] DEPRECATION WARNING: please use MorganGenerator
[04:14:50] DEPRECATION WARNING: please use MorganGenerat

Saved ensemble bundle to: catboost_ensemble_bundle.pkl


[04:14:53] DEPRECATION WARNING: please use MorganGenerator
[04:14:53] DEPRECATION WARNING: please use MorganGenerator
[04:14:53] DEPRECATION WARNING: please use MorganGenerator
[04:14:53] DEPRECATION WARNING: please use MorganGenerator
[04:14:53] DEPRECATION WARNING: please use MorganGenerator
[04:14:53] DEPRECATION WARNING: please use MorganGenerator
[04:14:53] DEPRECATION WARNING: please use MorganGenerator
[04:14:53] DEPRECATION WARNING: please use MorganGenerator
[04:14:53] DEPRECATION WARNING: please use MorganGenerator
[04:14:53] DEPRECATION WARNING: please use MorganGenerator
[04:14:53] DEPRECATION WARNING: please use MorganGenerator
[04:14:53] DEPRECATION WARNING: please use MorganGenerator
[04:14:53] DEPRECATION WARNING: please use MorganGenerator
[04:14:53] DEPRECATION WARNING: please use MorganGenerator
[04:14:53] DEPRECATION WARNING: please use MorganGenerator
[04:14:53] DEPRECATION WARNING: please use MorganGenerator
[04:14:53] DEPRECATION WARNING: please use MorganGenerat

In [12]:
def featurize_smiles_to_row(smiles, descriptor_columns, fp_radius, fp_nbits, fp_prefix="fp_"):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    row = {}

    # descriptors
    from rdkit.Chem import Descriptors, QED
    registry = {name: func for name, func in Descriptors.descList}
    registry["qed"] = QED.qed
    registry["QED"] = QED.qed

    for col in descriptor_columns:
        func = registry.get(col)
        if func is None:
            raise KeyError(f"Descriptor '{col}' not found in RDKit registry.")
        try:
            val = func(mol)
            if val is None or not np.isfinite(val):
                val = 0.0
        except Exception:
            val = 0.0
        row[col] = float(val)

    # fingerprints
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, fp_radius, nBits=fp_nbits)
    arr = np.zeros((fp_nbits,), dtype=np.int8)
    DataStructs.ConvertToNumpyArray(fp, arr)

    for i in range(fp_nbits):
        row[f"{fp_prefix}{i}"] = int(arr[i])

    return row


def max_tanimoto_similarity(query_bv, train_bitvects):
    sims = []
    for bv in train_bitvects:
        if bv is None:
            continue
        sims.append(DataStructs.TanimotoSimilarity(query_bv, bv))
    return max(sims) if sims else 0.0


def predict_with_uncertainty(smiles_list, bundle):
    descriptor_columns = bundle["descriptor_columns"]
    fp_radius = bundle["fp_radius"]
    fp_nbits = bundle["fp_nbits"]
    train_bitvects = bundle["train_bitvects"]
    ensemble_models = bundle["ensemble_models"]

    rows = []
    valid_mask = []
    query_bitvects = []

    for smi in smiles_list:
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            rows.append(None)
            valid_mask.append(False)
            query_bitvects.append(None)
            continue

        row = featurize_smiles_to_row(
            smi,
            descriptor_columns=descriptor_columns,
            fp_radius=fp_radius,
            fp_nbits=fp_nbits
        )
        rows.append(row)
        valid_mask.append(True)
        query_bitvects.append(
            AllChem.GetMorganFingerprintAsBitVect(mol, fp_radius, nBits=fp_nbits)
        )

    # build dataframe
    valid_rows = []
    valid_indices = []

    for i, row in enumerate(rows):
        if row is not None:
            valid_rows.append(row)
            valid_indices.append(i)

    results = []
    if valid_rows:
        Xq = pd.DataFrame(valid_rows)

        all_preds = []
        for model in ensemble_models:
            preds = model.predict(Xq)
            all_preds.append(preds)

        all_preds = np.vstack(all_preds)              # [n_models, n_samples]
        pred_mean = all_preds.mean(axis=0)
        pred_std = all_preds.std(axis=0)

        pred_map = {}
        for idx, mean_val, std_val in zip(valid_indices, pred_mean, pred_std):
            pred_map[idx] = (float(mean_val), float(std_val))

    for i, smi in enumerate(smiles_list):
        if not valid_mask[i]:
            results.append({
                "smiles": smi,
                "pred_mean": 0.0,
                "pred_std": 999.0,
                "max_similarity": 0.0,
                "novelty": 1.0,
                "is_valid": False
            })
            continue

        mean_val, std_val = pred_map[i]
        sim = max_tanimoto_similarity(query_bitvects[i], train_bitvects)

        results.append({
            "smiles": smi,
            "pred_mean": mean_val,
            "pred_std": std_val,
            "max_similarity": float(sim),
            "novelty": float(1.0 - sim),
            "is_valid": True
        })

    return pd.DataFrame(results)

In [13]:
with open("catboost_ensemble_bundle.pkl", "rb") as f:
    bundle = pickle.load(f)

test_smiles = [
    "CCO",
    "c1ccccc1",
    "CC(=O)Oc1ccccc1C(=O)O"
]

pred_df = predict_with_uncertainty(test_smiles, bundle)
print(pred_df)

                  smiles  pred_mean  pred_std  max_similarity   novelty  \
0                    CCO   3.793933  0.144486        0.173913  0.826087   
1               c1ccccc1   4.200979  0.151527        0.200000  0.800000   
2  CC(=O)Oc1ccccc1C(=O)O   3.700952  0.077090        0.365854  0.634146   

   is_valid  
0      True  
1      True  
2      True  


[04:15:16] DEPRECATION WARNING: please use MorganGenerator
[04:15:16] DEPRECATION WARNING: please use MorganGenerator
[04:15:16] DEPRECATION WARNING: please use MorganGenerator
[04:15:16] DEPRECATION WARNING: please use MorganGenerator
[04:15:16] DEPRECATION WARNING: please use MorganGenerator
[04:15:16] DEPRECATION WARNING: please use MorganGenerator


In [20]:
smiles = [
    "CCO",
    "c1ccccc1",
    "CC(=O)Oc1ccccc1C(=O)O",
]

with open(r"C:\Users\norep\Desktop\hakaton\test_smiles.smi", "w", encoding="utf-8", newline="\n") as f:
    f.write("\n".join(smiles))

In [27]:
import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import (
    train_test_split,
    KFold,
    GroupKFold,
    cross_validate
)
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_squared_error, make_scorer

from catboost import CatBoostRegressor
from xgboost import XGBRegressor
import optuna

from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold
from sklearn.base import BaseEstimator, RegressorMixin, clone


class SimpleAveragingRegressor(BaseEstimator, RegressorMixin):
    """
    Простой ансамбль для регрессии:
    обучает несколько моделей и усредняет их предсказания.
    Не зависит от VotingRegressor, поэтому не ломается
    на несовместимостях sklearn/xgboost tags.
    """
    def __init__(self, estimators, weights=None):
        self.estimators = estimators
        self.weights = weights

    def fit(self, X, y):
        self.fitted_estimators_ = []
        for name, est in self.estimators:
            fitted_est = clone(est)
            fitted_est.fit(X, y)
            self.fitted_estimators_.append((name, fitted_est))
        return self

    def predict(self, X):
        preds = []
        for _, est in self.fitted_estimators_:
            preds.append(np.asarray(est.predict(X), dtype=float))

        preds = np.vstack(preds)  # shape = (n_models, n_samples)

        if self.weights is None:
            return preds.mean(axis=0)

        weights = np.asarray(self.weights, dtype=float)
        weights = weights / weights.sum()
        return np.average(preds, axis=0, weights=weights)

# =========================================================
# CONFIG
# =========================================================
DATA_PATH = "data_rdkit.csv"
MODEL_BUNDLE_PATH = "model_comparison_bundle.pkl"

SMILES_COL = "Smiles"
TARGET_COL = "pIC50"

RANDOM_STATE = 42
TEST_SIZE = 0.2

# Чтобы не было слишком долго
N_SPLITS_RANDOM = 3
N_SPLITS_SCAFFOLD = 3
OPTUNA_TRIALS = 15

CATBOOST_PARAMS = dict(
    iterations=600,
    learning_rate=0.05,
    depth=6,
    loss_function="RMSE",
    eval_metric="R2",
    verbose=0,
    random_seed=RANDOM_STATE
)

XGB_PARAMS = dict(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    min_child_weight=1,
    objective="reg:squarederror",
    tree_method="hist",
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbosity=0
)


# =========================================================
# UTILS
# =========================================================
class ColumnSelector(BaseEstimator, TransformerMixin):
    """
    Выбирает только нужные колонки и сохраняет порядок признаков.
    """
    def __init__(self, columns):
        self.columns = columns

    def fit(self, X, y=None):
        missing = [c for c in self.columns if c not in X.columns]
        if missing:
            raise ValueError(f"Missing columns in input DataFrame: {missing}")
        return self

    def transform(self, X):
        return X.loc[:, self.columns]


def rmse_func(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))


rmse_scorer = make_scorer(rmse_func, greater_is_better=False)


def get_scaffold(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return MurckoScaffold.MurckoScaffoldSmiles(mol=mol)


def build_catboost_pipeline(feature_columns, params):
    return Pipeline([
        ("select_columns", ColumnSelector(feature_columns)),
        ("model", CatBoostRegressor(**params))
    ])


def build_xgb_pipeline(feature_columns, params):
    return Pipeline([
        ("select_columns", ColumnSelector(feature_columns)),
        ("model", XGBRegressor(**params))
    ])


def build_ensemble_pipeline(feature_columns, cat_params, xgb_params):
    return Pipeline([
        ("select_columns", ColumnSelector(feature_columns)),
        ("model", SimpleAveragingRegressor([
            ("cat", CatBoostRegressor(**cat_params)),
            ("xgb", XGBRegressor(**xgb_params))
        ]))
    ])


def evaluate_model(name, model, X_data, y_data, cv, groups=None):
    scoring = {
        "r2": "r2",
        "rmse": rmse_scorer
    }

    result = cross_validate(
        model,
        X_data,
        y_data,
        cv=cv,
        groups=groups,
        scoring=scoring,
        n_jobs=1,
        return_train_score=False
    )

    r2_mean = float(result["test_r2"].mean())
    r2_std = float(result["test_r2"].std())
    rmse_mean = float(-result["test_rmse"].mean())
    rmse_std = float(result["test_rmse"].std())

    print(f"\n{name}")
    print(f"R2 mean:   {r2_mean:.4f} ± {r2_std:.4f}")
    print(f"RMSE mean: {rmse_mean:.4f} ± {rmse_std:.4f}")

    return {
        "model": name,
        "r2_mean": r2_mean,
        "r2_std": r2_std,
        "rmse_mean": rmse_mean,
        "rmse_std": rmse_std
    }


# =========================================================
# LOAD DATA
# =========================================================
df = pd.read_csv(DATA_PATH)
df = df.dropna().copy()

df["scaffold"] = df[SMILES_COL].apply(get_scaffold)
df = df.dropna(subset=["scaffold"]).copy()

feature_columns = [c for c in df.columns if c not in [SMILES_COL, TARGET_COL, "scaffold"]]

X = df[feature_columns].copy()
y = df[TARGET_COL].values
groups = df["scaffold"].values

print("=" * 60)
print("DATA SUMMARY")
print("=" * 60)
print(f"Rows: {len(df)}")
print(f"Features: {len(feature_columns)}")
print(f"Unique scaffolds: {df['scaffold'].nunique()}")
print("Feature columns:")
print(feature_columns)

# =========================================================
# TRAIN / TEST SPLIT
# =========================================================
X_train, X_test, y_train, y_test, groups_train, groups_test = train_test_split(
    X, y, groups,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE
)

print("\n" + "=" * 60)
print("TRAIN / TEST SPLIT")
print("=" * 60)
print(f"Train size: {len(X_train)}")
print(f"Test size:  {len(X_test)}")

# =========================================================
# PIPELINES
# =========================================================
cat_pipeline = build_catboost_pipeline(feature_columns, CATBOOST_PARAMS)
xgb_pipeline = build_xgb_pipeline(feature_columns, XGB_PARAMS)
ensemble_pipeline = build_ensemble_pipeline(feature_columns, CATBOOST_PARAMS, XGB_PARAMS)

# =========================================================
# RANDOM CV
# =========================================================
print("\n" + "=" * 60)
print("RANDOM CV")
print("=" * 60)

random_cv = KFold(n_splits=N_SPLITS_RANDOM, shuffle=True, random_state=RANDOM_STATE)

results_random = []
results_random.append(evaluate_model("CatBoost", cat_pipeline, X_train, y_train, random_cv))
results_random.append(evaluate_model("XGBoost", xgb_pipeline, X_train, y_train, random_cv))
results_random.append(evaluate_model("Ensemble", ensemble_pipeline, X_train, y_train, random_cv))

random_results_df = pd.DataFrame(results_random).sort_values("r2_mean", ascending=False)
print("\nRandom CV summary:")
print(random_results_df)

# =========================================================
# SCAFFOLD CV
# =========================================================
print("\n" + "=" * 60)
print("SCAFFOLD CV")
print("=" * 60)

scaffold_cv = GroupKFold(n_splits=N_SPLITS_SCAFFOLD)

results_scaffold = []
results_scaffold.append(
    evaluate_model("CatBoost", cat_pipeline, X_train, y_train, scaffold_cv, groups=groups_train)
)
results_scaffold.append(
    evaluate_model("XGBoost", xgb_pipeline, X_train, y_train, scaffold_cv, groups=groups_train)
)
results_scaffold.append(
    evaluate_model("Ensemble", ensemble_pipeline, X_train, y_train, scaffold_cv, groups=groups_train)
)

scaffold_results_df = pd.DataFrame(results_scaffold).sort_values("r2_mean", ascending=False)
print("\nScaffold CV summary:")
print(scaffold_results_df)

# =========================================================
# OPTUNA FOR XGBOOST ONLY
# =========================================================
print("\n" + "=" * 60)
print("OPTUNA: XGBOOST")
print("=" * 60)

def objective_xgb(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 150, 500, step=50),
        "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.15, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 8),
        "subsample": trial.suggest_float("subsample", 0.7, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.7, 1.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-2, 10.0, log=True),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "objective": "reg:squarederror",
        "tree_method": "hist",
        "random_state": RANDOM_STATE,
        "n_jobs": -1,
        "verbosity": 0
    }

    pipeline = build_xgb_pipeline(feature_columns, params)

    cv = GroupKFold(n_splits=N_SPLITS_SCAFFOLD)

    scores = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        groups=groups_train,
        scoring={"r2": "r2"},
        n_jobs=1,
        return_train_score=False
    )

    return float(scores["test_r2"].mean())


study = optuna.create_study(direction="maximize")
study.optimize(objective_xgb, n_trials=OPTUNA_TRIALS, show_progress_bar=False)

print(f"Best Optuna CV R2: {study.best_value:.4f}")
print("Best params:")
print(json.dumps(study.best_params, indent=2))

best_xgb_params = study.best_params.copy()
best_xgb_params.update({
    "objective": "reg:squarederror",
    "tree_method": "hist",
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
    "verbosity": 0
})

best_xgb_pipeline = build_xgb_pipeline(feature_columns, best_xgb_params)
tuned_ensemble_pipeline = build_ensemble_pipeline(feature_columns, CATBOOST_PARAMS, best_xgb_params)

# =========================================================
# COMPARE TUNED MODELS ON SCAFFOLD CV
# =========================================================
print("\n" + "=" * 60)
print("POST-OPTUNA SCAFFOLD CV")
print("=" * 60)

post_optuna_results = []
post_optuna_results.append(
    evaluate_model("CatBoost", cat_pipeline, X_train, y_train, scaffold_cv, groups=groups_train)
)
post_optuna_results.append(
    evaluate_model("XGBoost_tuned", best_xgb_pipeline, X_train, y_train, scaffold_cv, groups=groups_train)
)
post_optuna_results.append(
    evaluate_model("Ensemble_tuned", tuned_ensemble_pipeline, X_train, y_train, scaffold_cv, groups=groups_train)
)

post_optuna_df = pd.DataFrame(post_optuna_results).sort_values("r2_mean", ascending=False)
print("\nPost-Optuna scaffold CV summary:")
print(post_optuna_df)

best_model_name = post_optuna_df.iloc[0]["model"]
print(f"\nBest model by scaffold CV: {best_model_name}")

if best_model_name == "CatBoost":
    final_pipeline = cat_pipeline
elif best_model_name == "XGBoost_tuned":
    final_pipeline = best_xgb_pipeline
else:
    final_pipeline = tuned_ensemble_pipeline

# =========================================================
# FIT FINAL MODEL
# =========================================================
print("\n" + "=" * 60)
print("FIT FINAL MODEL")
print("=" * 60)

final_pipeline.fit(X_train, y_train)

# =========================================================
# TEST EVALUATION
# =========================================================
y_pred = final_pipeline.predict(X_test)

r2 = r2_score(y_test, y_pred)
rmse = rmse_func(y_test, y_pred)

print("\n" + "=" * 60)
print("FINAL TEST METRICS")
print("=" * 60)
print(f"Model: {best_model_name}")
print(f"R2 Score: {r2:.4f}")
print(f"RMSE:     {rmse:.4f}")

# =========================================================
# FEATURE IMPORTANCE
# =========================================================
print("\n" + "=" * 60)
print("FEATURE IMPORTANCE")
print("=" * 60)

importance_df = None

try:
    if best_model_name == "CatBoost":
        model = final_pipeline.named_steps["model"]
        importances = model.get_feature_importance()

    elif best_model_name == "XGBoost_tuned":
        model = final_pipeline.named_steps["model"]
        importances = model.feature_importances_

    else:
        # Для ансамбля усредняем importance двух моделей
        ensemble_model = final_pipeline.named_steps["model"]

        fitted_dict = dict(ensemble_model.fitted_estimators_)
        cat_model = fitted_dict["cat"]
        xgb_model = fitted_dict["xgb"]

        cat_imp = cat_model.get_feature_importance()
        xgb_imp = xgb_model.feature_importances_

        importances = (np.array(cat_imp) + np.array(xgb_imp)) / 2.0

    importance_df = pd.DataFrame({
        "feature": feature_columns,
        "importance": importances
    }).sort_values("importance", ascending=False)

    print(importance_df)

except Exception as e:
    print(f"Could not compute feature importance: {e}")

# =========================================================
# SAVE BUNDLE
# =========================================================
bundle = {
    "final_pipeline": final_pipeline,
    "best_model_name": best_model_name,
    "feature_columns": feature_columns,
    "smiles_col": SMILES_COL,
    "target_col": TARGET_COL,
    "catboost_params": CATBOOST_PARAMS,
    "xgb_base_params": XGB_PARAMS,
    "xgb_best_params": best_xgb_params,
    "random_cv_results": random_results_df.to_dict(orient="records"),
    "scaffold_cv_results": scaffold_results_df.to_dict(orient="records"),
    "post_optuna_scaffold_results": post_optuna_df.to_dict(orient="records"),
    "test_metrics": {
        "r2": float(r2),
        "rmse": float(rmse)
    },
    "feature_importance": None if importance_df is None else importance_df.to_dict(orient="records"),
    "logic_note": (
        "Comparison of CatBoost, XGBoost, and their ensemble. "
        "Model selection is based on scaffold CV on train set only. "
        "Then best model is fitted on X_train and evaluated on X_test."
    )
}

with open(MODEL_BUNDLE_PATH, "wb") as f:
    pickle.dump(bundle, f)

print(f"\nSaved bundle to: {MODEL_BUNDLE_PATH}")

DATA SUMMARY
Rows: 12358
Features: 164
Unique scaffolds: 4735
Feature columns:
['MaxAbsEStateIndex', 'MinAbsEStateIndex', 'MinEStateIndex', 'qed', 'SPS', 'MolWt', 'MaxPartialCharge', 'MinPartialCharge', 'FpDensityMorgan1', 'BCUT2D_MWHI', 'BCUT2D_MWLOW', 'BCUT2D_CHGHI', 'BCUT2D_CHGLO', 'BCUT2D_LOGPHI', 'BCUT2D_LOGPLOW', 'BCUT2D_MRHI', 'BCUT2D_MRLOW', 'AvgIpc', 'BalabanJ', 'BertzCT', 'HallKierAlpha', 'PEOE_VSA1', 'PEOE_VSA10', 'PEOE_VSA11', 'PEOE_VSA12', 'PEOE_VSA13', 'PEOE_VSA14', 'PEOE_VSA2', 'PEOE_VSA3', 'PEOE_VSA4', 'PEOE_VSA5', 'PEOE_VSA6', 'PEOE_VSA7', 'PEOE_VSA8', 'PEOE_VSA9', 'SMR_VSA1', 'SMR_VSA10', 'SMR_VSA2', 'SMR_VSA3', 'SMR_VSA4', 'SMR_VSA5', 'SMR_VSA6', 'SMR_VSA7', 'SMR_VSA9', 'SlogP_VSA1', 'SlogP_VSA10', 'SlogP_VSA11', 'SlogP_VSA12', 'SlogP_VSA2', 'SlogP_VSA3', 'SlogP_VSA4', 'SlogP_VSA5', 'SlogP_VSA7', 'SlogP_VSA8', 'TPSA', 'EState_VSA1', 'EState_VSA10', 'EState_VSA11', 'EState_VSA2', 'EState_VSA3', 'EState_VSA4', 'EState_VSA5', 'EState_VSA6', 'EState_VSA7', 'EState_VSA8',

[I 2026-03-22 08:16:05,467] A new study created in memory with name: no-name-d5500298-c610-46a0-89dd-4a7bcc119541



Ensemble
R2 mean:   0.5962 ± 0.0428
RMSE mean: 0.9280 ± 0.0884

Scaffold CV summary:
      model   r2_mean    r2_std  rmse_mean  rmse_std
1   XGBoost  0.601357  0.044472   0.922117  0.091926
2  Ensemble  0.596180  0.042812   0.927951  0.088385
0  CatBoost  0.576921  0.040835   0.949632  0.083346

OPTUNA: XGBOOST


[I 2026-03-22 08:16:21,821] Trial 0 finished with value: 0.6062022781893932 and parameters: {'n_estimators': 350, 'learning_rate': 0.02859359272127658, 'max_depth': 8, 'subsample': 0.740056047990857, 'colsample_bytree': 0.9242011082310596, 'reg_lambda': 0.11661960727561252, 'min_child_weight': 4}. Best is trial 0 with value: 0.6062022781893932.
[I 2026-03-22 08:16:33,219] Trial 1 finished with value: 0.6033575771502971 and parameters: {'n_estimators': 400, 'learning_rate': 0.11824966190784224, 'max_depth': 7, 'subsample': 0.8655901696634891, 'colsample_bytree': 0.7442134272213573, 'reg_lambda': 0.08599626365348054, 'min_child_weight': 2}. Best is trial 0 with value: 0.6062022781893932.
[I 2026-03-22 08:16:42,873] Trial 2 finished with value: 0.5601431342492579 and parameters: {'n_estimators': 500, 'learning_rate': 0.02137963141520068, 'max_depth': 5, 'subsample': 0.9541097821199493, 'colsample_bytree': 0.9119280818318547, 'reg_lambda': 5.655271110181195, 'min_child_weight': 1}. Best is

Best Optuna CV R2: 0.6090
Best params:
{
  "n_estimators": 400,
  "learning_rate": 0.042898594109802124,
  "max_depth": 7,
  "subsample": 0.774490263746469,
  "colsample_bytree": 0.9488943811233241,
  "reg_lambda": 0.06554703728397404,
  "min_child_weight": 10
}

POST-OPTUNA SCAFFOLD CV

CatBoost
R2 mean:   0.5769 ± 0.0408
RMSE mean: 0.9496 ± 0.0833

XGBoost_tuned
R2 mean:   0.6090 ± 0.0391
RMSE mean: 0.9134 ± 0.0867

Ensemble_tuned
R2 mean:   0.6023 ± 0.0405
RMSE mean: 0.9209 ± 0.0862

Post-Optuna scaffold CV summary:
            model   r2_mean    r2_std  rmse_mean  rmse_std
1   XGBoost_tuned  0.609026  0.039092   0.913367  0.086685
2  Ensemble_tuned  0.602337  0.040534   0.920918  0.086231
0        CatBoost  0.576921  0.040835   0.949632  0.083346

Best model by scaffold CV: XGBoost_tuned

FIT FINAL MODEL

FINAL TEST METRICS
Model: XGBoost_tuned
R2 Score: 0.7678
RMSE:     0.6951

FEATURE IMPORTANCE
               feature  importance
5                MolWt    0.052087
155         fr_